In [36]:
!pip install yfinance
!pip install nbformat ipykernel


[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.2 -> 25.3
[notice] To update, run: pip install --upgrade pip


## Imports

In [37]:
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

pd.set_option('display.width', 160)
pd.set_option('display.max_columns',50)
plt.rcParams['figure.figsize'] = (10,5)

## Globals

In [38]:
TICKERS = ['TSLA','NVDA', 'META', '^GSPC']
START = '2018-01-01'
END = '2025-10-31'
INTERVAL = '1d'
print("tickers:", TICKERS, "start:", START, "end:", END, "interval:", INTERVAL)

tickers: ['TSLA', 'NVDA', 'META', '^GSPC'] start: 2018-01-01 end: 2025-10-31 interval: 1d


## Load Data

In [39]:
def load_yfinance_data(
        tickers,
        start,
        end,
        interval="1d",
        auto_adjust=True,
        group_by='ticker',
        progress=False,
        threads=True,
        **download_kwargs,
) -> pd.DataFrame:
    """
    Load historical price data from Yahoo Finance using yfinance library.
    """

    print(
        f"Downloading data for tickers={tickers}, "
        f"start={start}, end={end}, interval={interval}"
    )

    df = yf.download(
        tickers=tickers,
        start=start,
        end=end,
        interval=interval,
        auto_adjust=auto_adjust,
        group_by=group_by,
        progress=progress,
        threads=threads,
        **download_kwargs,
    )

    df.index = pd.to_datetime(df.index).tz_localize(None)
    print("Columns type:", type(df.columns), "Shape:", df.shape)
    return df

In [40]:
df = load_yfinance_data(TICKERS, start=START, end=END, interval=INTERVAL)
df.head()

Columns type: <class 'pandas.core.indexes.multi.MultiIndex'> Shape: (1969, 20)


Ticker           TSLA                                                  NVDA                                                 META                          \
Price            Open       High        Low      Close     Volume      Open      High       Low     Close     Volume        Open        High         Low   
Date                                                                                                                                                       
2018-01-02  20.799999  21.474001  20.733334  21.368668   65283000  4.840540  4.932515  4.808893  4.928806  355616000  176.591351  180.467464  176.462157   
2018-01-03  21.400000  21.683332  21.036667  21.150000   67822500  5.046248  5.283602  5.037594  5.253191  914704000  180.765644  183.647870  180.219011   
2018-01-04  20.858000  21.236668  20.378668  20.974667  149194500  5.334534  5.391152  5.258629  5.280881  583268000  183.767107  185.069093  182.972021   
2018-01-05  21.108000  21.149332  20.799999  21.105333   68868000  5.295717  5.362967  5.218824  5.325634  580124000  184.452880  185.754851  183.796920   
2018-01-08  21.066668  22.468000  21.033333  22.427334  147891000  5.449257  5.562989  5.404258  5.488816  881216000  186.053019  187.742600  185.188355   

Ticker                                  ^GSPC                                                     
Price            Close    Volume         Open         High          Low        Close      Volume  
Date                                                                                              
2018-01-02  180.308441  18151900  2683.729980  2695.889893  2682.360107  2695.810059  3397430000  
2018-01-03  183.538544  16886600  2697.850098  2714.370117  2697.770020  2713.060059  3544030000  
2018-01-04  183.200607  13880900  2719.310059  2729.290039  2719.070068  2723.989990  3697340000  
2018-01-05  185.705170  13574500  2731.330078  2743.449951  2727.919922  2743.149902  3239280000  
2018-01-08  187.126404  17994700  2742.669922  2748.510010  2737.600098  2747.709961  3246160000

In [41]:
print('Index range:', df.index.min(), "→", df.index.max(), "  rows:", len(df))

Index range: 2018-01-02 00:00:00 → 2025-10-30 00:00:00   rows: 1969


In [42]:
print("\nColumns preview:")
print(df.columns[:12])



Columns preview:
MultiIndex([('TSLA',   'Open'),
            ('TSLA',   'High'),
            ('TSLA',    'Low'),
            ('TSLA',  'Close'),
            ('TSLA', 'Volume'),
            ('NVDA',   'Open'),
            ('NVDA',   'High'),
            ('NVDA',    'Low'),
            ('NVDA',  'Close'),
            ('NVDA', 'Volume'),
            ('META',   'Open'),
            ('META',   'High')],
           names=['Ticker', 'Price'])


In [43]:
print("\n.info():")
df.info()


.info():
<class 'pandas.core.frame.DataFrame'>
DatetimeIndex: 1969 entries, 2018-01-02 to 2025-10-30
Data columns (total 20 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   (TSLA, Open)     1969 non-null   float64
 1   (TSLA, High)     1969 non-null   float64
 2   (TSLA, Low)      1969 non-null   float64
 3   (TSLA, Close)    1969 non-null   float64
 4   (TSLA, Volume)   1969 non-null   int64  
 5   (NVDA, Open)     1969 non-null   float64
 6   (NVDA, High)     1969 non-null   float64
 7   (NVDA, Low)      1969 non-null   float64
 8   (NVDA, Close)    1969 non-null   float64
 9   (NVDA, Volume)   1969 non-null   int64  
 10  (META, Open)     1969 non-null   float64
 11  (META, High)     1969 non-null   float64
 12  (META, Low)      1969 non-null   float64
 13  (META, Close)    1969 non-null   float64
 14  (META, Volume)   1969 non-null   int64  
 15  (^GSPC, Open)    1969 non-null   float64
 16  (^GSPC, High)    1969 non-null  

## Analysis Functions

In [44]:
def extract_close_prices(df: pd.DataFrame, tickers: list[str] | None = None) -> pd.DataFrame:
    """
    Return a DataFrame with closing prices for specified tickers.
    """
    if isinstance(df.columns, pd.MultiIndex):
        closes=df.xs('Close', axis=1, level=1)
    else:
        if "Close" not in df.columns:
            raise ValueError("DataFrame does not contain 'Close' prices.")
        name=tickers[0] if tickers else 'Close'
        closes=df["Close"].to_frame(name=name)
    closes=closes.sort_index()
    closes.index=pd.to_datetime(closes.index).tz_localize(None)
    return closes

In [45]:
def clean_closes(daily_closing: pd.DataFrame) -> tuple[pd.DataFrame, pd.Series]:
    """
    Forward-fills gaps, drops leading all-NA rows, and returns:
    - closes_ffill: cleaned prices
    - na_counts: NA counts *before* filling (for reporting)
    """
    na_counts=daily_closing.isna().sum()
    print("NA per ticker:\n", na_counts)

    closes_ffill=daily_closing.ffill().dropna(how='all')
    closes_ffill=closes_ffill.sort_index()
    print("After ffill → shape:", closes_ffill.shape)

    return closes_ffill, na_counts

In [46]:
def compute_daily_returns(df_closing: pd.DataFrame) -> pd.DataFrame:
    """Compute daily returns from closing prices DataFrame."""
    returns=df_closing.pct_change().dropna(how='all')
    return returns

In [47]:
def monthly_aggregate(daily_closing: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Resample to month-end frequency and compute % monthly returns
    Returns monthly_close, monthly_return_percent
    """
    monthly_close=daily_closing.resample('ME').last()
    monthly_return=monthly_close.pct_change().dropna(how='all') * 100.0
    return monthly_close, monthly_return

In [48]:
def rolling_statistics(
        df_returns: pd.DataFrame,
        window: int = 30,
        trading_days_per_year: int = 252) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compute rolling mean and rolling volatility (annualized) of daily returns.
    Returns rolling_mean, rolling_volatility
    """
    rolling_mean = df_returns.rolling(window).mean()
    rolling_volatility = df_returns.rolling(window).std() * np.sqrt(trading_days_per_year)
    return rolling_mean, rolling_volatility

In [49]:
def compute_drawdown(df_return: pd.DataFrame) -> pd.DataFrame:
    """
    Compute drawdown series from price series.
    drawdown_t = (price_t / running_max_t) - 1
    """
    cum_max = df_return.cummax(axis=0)
    drawdown = df_return / cum_max - 1.0
    return drawdown

In [50]:
def total_return_index(daily_closing: pd.DataFrame) -> pd.DataFrame:
    """
    Build a Total Return Index (TRI) daily closing prices DataFrame
    so TRI is just cumulative product of (1 + daily return), equivalently close / first close.
    """
    return (1.0 + daily_closing.pct_change()).cumprod().fillna(1.0)

In [51]:
def cagr(series: pd.Series) -> float:
    """
    Compound Annual Growth Rate, computed from the first and last values of a *level* series.
    """
    s = series.dropna()
    if s.empty:
        return np.nan
    years = (s.index[-1] - s.index[0]).days / 365.25
    return (s.iloc[-1] / s.iloc[0])**(1/years) - 1 if years > 0 else np.nan



In [52]:
def max_drawdown(series: pd.Series) -> float:
    """
    Most negative drawdown value (depth) from a *level* series.
    """
    max_drawdown = compute_drawdown(series)
    return float(max_drawdown.min())

In [53]:
def var_cvar(returns: pd.Series, alpha: float = 0.05) -> tuple[float, float]:
    """
    Var: Value at Risk; CVaR: Conditional Value at Risk.
    Historical (non-parametric) VaR/CVaR on a return *series*.
    VaR_alpha = alpha-quantile; CVaR = mean of the tail <= VaR.
    """
    r = returns.dropna().sort_values()
    if r.empty:
        return np.nan, np.nan
    var = float(r.quantile(alpha))
    cvar = float(r[r <= var].mean()) if (r <= var).any() else np.nan
    return var, cvar

In [54]:
def build_performance_table(
    daily_closing: pd.DataFrame,
    tri: pd.DataFrame,
    alpha: float = 0.05
) -> pd.DataFrame:
    """
    Build a tidy summary table with per-ticker:
      CAGR, Vol (annual), MaxDD, VaR5%, CVaR5% (daily metrics).
    """
    rets_d = daily_closing.pct_change()
    rows = []
    for t in tri.columns:
        tri_t = tri[t]
        r     = rets_d[t]
        vr, cvr = var_cvar(r, alpha=alpha)
        rows.append({
            "Ticker": t,
            "CAGR": cagr(tri_t),
            "Vol (annual)": float(r.std(ddof=1) * np.sqrt(252)),
            "MaxDD": max_drawdown(tri_t),
            f"VaR{int(alpha*100)}% (daily)": vr,
            f"CVaR{int(alpha*100)}% (daily)": cvr,
        })
    return pd.DataFrame(rows).set_index("Ticker")

In [55]:
def correlation_matrices(
    daily_returns: pd.DataFrame,
    monthly_returns_percent: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Pearson correlation matrices for daily and monthly returns.
    (Monthly input expected in percent — we convert back to fraction before corr.)
    """
    corr_daily   = daily_returns.corr()
    corr_monthly = (monthly_returns_percent / 100.0).corr()
    return corr_daily, corr_monthly

In [56]:
def rolling_beta(
    asset_returns: pd.Series,
    market_returns: pd.Series,
    window: int = 252
) -> pd.Series:
    """
    Rolling beta = Cov(asset, market) / Var(market), over a rolling window (trading days).
    """
    cov = asset_returns.rolling(window).cov(market_returns)
    var = market_returns.rolling(window).var()
    return cov / var

## Plots

In [57]:
def plot_normalized(tri: pd.DataFrame, title: str = "Normalized Total Return (start=100)"):
    norm = tri / tri.iloc[0] * 100.0
    fig = px.line(norm, title=title, labels={"value":"Index Level","index":"Date","variable":"Ticker"})
    fig.show()


def plot_drawdowns(levels: pd.DataFrame, title: str = "Drawdowns (from running peak)"):
    dd = compute_drawdown(levels)
    fig = px.line(dd, title=title, labels={"value":"Drawdown","index":"Date","variable":"Ticker"})
    fig.update_yaxes(tickformat=".0%")
    fig.show()


def plot_rolling_vol(returns: pd.DataFrame, window: int = 30, title: str | None = None):
    rvol = returns.rolling(window).std() * np.sqrt(252)
    ttl = title or f"Rolling Volatility ({window}d, annualized)"
    fig = px.line(rvol, title=ttl, labels={"value":"Volatility","index":"Date","variable":"Ticker"})
    fig.show()


def plot_corr_heatmap(corr: pd.DataFrame, title: str = "Correlation Heatmap"):
    fig = px.imshow(corr, text_auto=True, aspect="auto", color_continuous_scale="RdBu", origin="lower",
                    title=title)
    fig.update_traces(zmin=-1, zmax=1)
    fig.show()


def plot_risk_return_scatter(summary: pd.DataFrame, title: str = "Risk–Return: CAGR vs Annualized Volatility"):
    fig = px.scatter(
        summary.reset_index(),
        x="Vol (annual)", y="CAGR", text="Ticker", color="Ticker",
        title=title, labels={"x":"Vol (annual)","y":"CAGR"}
    )
    fig.update_traces(textposition="top center")
    fig.show()


def plot_tail_risk_bars(summary: pd.DataFrame, alpha: float = 0.05):
    var_col  = f"VaR{int(alpha*100)}% (daily)"
    cvar_col = f"CVaR{int(alpha*100)}% (daily)"
    sub = summary[[var_col, cvar_col]].reset_index()
    fig = go.Figure(data=[
        go.Bar(name=var_col,  x=sub["Ticker"], y=sub[var_col]),
        go.Bar(name=cvar_col, x=sub["Ticker"], y=sub[cvar_col]),
    ])
    fig.update_layout(barmode="group", title="Tail Risk (historical)")
    fig.show()


def plot_hist_daily_returns(daily_returns: pd.DataFrame, facet_wrap: int = 2):
    """
    Daily returns are FRACTIONS (e.g., 0.01 = 1%).
    Faceted histograms, one per ticker.
    """
    long = daily_returns.melt(ignore_index=False, var_name="Ticker", value_name="Return")
    n = long["Return"].dropna().size
    nbins = max(10, int(np.sqrt(n)))  # simple √n rule
    fig = px.histogram(
        long, x="Return", facet_col="Ticker", facet_col_wrap=facet_wrap,
        nbins=nbins, title="Histograms of Daily Returns"
    )
    fig.update_xaxes(tickformat=".1%")
    fig.show()


def plot_hist_monthly_returns(monthly_return_pct: pd.DataFrame, facet_wrap: int = 2):
    """
    Monthly returns are already in PERCENT (% units) in your pipeline.
    """
    long = monthly_return_pct.melt(ignore_index=False, var_name="Ticker", value_name="ReturnPct")
    n = long["ReturnPct"].dropna().size
    nbins = max(10, int(np.sqrt(n)))
    fig = px.histogram(
        long, x="ReturnPct", facet_col="Ticker", facet_col_wrap=facet_wrap,
        nbins=nbins, title="Histograms of Monthly Returns (%)"
    )
    fig.show()

## Run Analysis

In [58]:
# Extract and Clean the Data
closes = extract_close_prices(df, tickers=TICKERS)
daily_closing, na_counts = clean_closes(closes)
print("NA per ticker before fill:\n", na_counts)
print("After ffill ->", daily_closing.shape)

NA per ticker:
 Ticker
TSLA     0
NVDA     0
META     0
^GSPC    0
dtype: int64
After ffill → shape: (1969, 4)
NA per ticker before fill:
 Ticker
TSLA     0
NVDA     0
META     0
^GSPC    0
dtype: int64
After ffill -> (1969, 4)


In [59]:
# Aggregations & Returns
monthly_closing, monthly_return_percent = monthly_aggregate(daily_closing)
daily_returns = compute_daily_returns(daily_closing)
rolling_mean, rolling_volatility = rolling_statistics(daily_returns, window=30, trading_days_per_year=252)

In [60]:
# Build Total Return Index
total_return_idx = total_return_index(daily_closing)
total_return_idx.head()

Ticker,TSLA,NVDA,META,^GSPC
Date,,,,
2018-01-02,1.000000,1.000000,1.000000,1.000000
2018-01-03,0.989767,1.065814,1.017914,1.006399
2018-01-04,0.981562,1.071432,1.016040,1.010453
2018-01-05,0.987677,1.080512,1.029931,1.017561
2018-01-08,1.049543,1.113620,1.037813,1.019252


In [61]:
# Performance Summary
# non_index_cols = [t for t in total_return_idx.columns if not t.startswith("^")]
non_index_cols = [t for t in total_return_idx.columns]
summary_table = build_performance_table(
    daily_closing=daily_closing[non_index_cols],
    tri=total_return_idx[non_index_cols],
    alpha=0.05
)
summary_table["Sharpe (rf=0)"] = summary_table["CAGR"] / summary_table["Vol (annual)"]
display(summary_table)

,CAGR,Vol (annual),MaxDD,VaR5% (daily),CVaR5% (daily),Sharpe (rf=0)
Ticker,,,,,,
TSLA,0.471971,0.640136,-0.736322,-0.058386,-0.087189,0.737298
NVDA,0.608178,0.515603,-0.663351,-0.047782,-0.071011,1.179547
META,0.181843,0.417262,-0.767361,-0.037528,-0.058766,0.435801
^GSPC,0.125989,0.197317,-0.339250,-0.018386,-0.030152,0.638511


In [62]:
# Correlations
corr_daily, corr_monthly = correlation_matrices(daily_returns[non_index_cols], monthly_return_percent[non_index_cols])
print("Daily returns correlation matrix:", corr_daily.round(3))
print("\nMonthly returns correlation matrix:", corr_monthly.round(3))

Daily returns correlation matrix: Ticker   TSLA   NVDA   META  ^GSPC
Ticker                            
TSLA    1.000  0.444  0.339  0.501
NVDA    0.444  1.000  0.528  0.697
META    0.339  0.528  1.000  0.624
^GSPC   0.501  0.697  0.624  1.000

Monthly returns correlation matrix: Ticker   TSLA   NVDA   META  ^GSPC
Ticker                            
TSLA    1.000  0.354  0.280  0.479
NVDA    0.354  1.000  0.525  0.632
META    0.280  0.525  1.000  0.562
^GSPC   0.479  0.632  0.562  1.000


In [63]:
# Plots
import plotly.io as pio
pio.renderers.default = "vscode"

plot_hist_daily_returns(daily_returns[non_index_cols])
plot_hist_monthly_returns(monthly_return_percent[non_index_cols])


In [64]:
plot_normalized(total_return_idx[non_index_cols], title="Normalized Total Return Index (start=100)")


In [65]:
plot_drawdowns(total_return_idx[non_index_cols], title="Drawdowns of Total Return Index")


In [66]:
plot_rolling_vol(daily_returns[non_index_cols], window=30, title="30-day Rolling Volatility (annualized)")


In [67]:
plot_corr_heatmap(corr_daily, title="Daily Returns Correlation Heatmap")


In [68]:
plot_corr_heatmap(corr_monthly, title="Monthly Returns Correlation Heatmap")


In [69]:
plot_risk_return_scatter(summary_table, title="Risk–Return: CAGR vs Annualized Volatility")


In [70]:
plot_tail_risk_bars(summary_table, alpha=0.05)